# EXP-20260828-integrated-01

```text
실험 ID: EXP-20260828-integrated-01
생성 기준일: 2026-08-28
기준 소스: src 최신본
실험 목적: Graph·Vector 단독 검증 결과를 Agent 흐름(Q02·Q03·Q05)에 연결
```

이 노트북은 생성 당시 `src/` 의 복사본을 가진 독립 실험 공간이다 (계획 v4 §3).
`%%module` 셀 수정은 `src/` 에 자동 반영되지 않으며, `sync_to_py(dry_run=False)` 는 최종 채택 시에만 실행한다.

**전제**: `agent_graph_0828.ipynb`(G01~G12)와 `agent_vector_0828.ipynb`(V01~V08) 단독 검증 통과 후 실행한다.
RDB 단계는 LogicalPlan compiler 가 IN-list 후보 랭킹을 지원하지 않아 실험에서는 read-only 직접 SQL 로 확인한다
(승격 시 binding `etf_kr.net_assets` 경로로 전환).

In [ ]:
# === 셀 매직 정의: 각 셀을 실제 모듈로 등록한다 ===
# 사용법: 셀 첫 줄에 `%%module <모듈명> <src 기준 경로>`.
# 셀을 수정하고 재실행하면 sys.modules 가 교체되므로,
# 그 모듈을 import 하는 하위 셀들을 다시 실행하면 수정본이 반영된다.
import json as _json
import sys as _sys
import types as _types
from pathlib import Path

from IPython.core.magic import register_cell_magic

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
NB_PATH = REPO_ROOT / "test" / "notebook" / "experiments" / "agent_integrated_0828.ipynb"
RESULTS_DIR = REPO_ROOT / "test" / "notebook" / "experiments" / "results"
RESULTS_DIR.mkdir(exist_ok=True)


@register_cell_magic("module")
def _module_magic(line, cell):
    name, relpath = line.split()
    mod = _types.ModuleType(name)
    mod.__file__ = str(REPO_ROOT / "src" / relpath)
    _sys.modules[name] = mod
    parts = name.split(".")
    for i in range(1, len(parts)):
        pkg = ".".join(parts[:i])
        parent = _sys.modules.setdefault(pkg, _types.ModuleType(pkg))
        setattr(parent, parts[i], _sys.modules.get(name) if i == len(parts) - 1 else _sys.modules.setdefault(".".join(parts[:i + 1]), _types.ModuleType(".".join(parts[:i + 1]))))
    exec(compile(cell, mod.__file__, "exec"), mod.__dict__)
    print(f"registered: {name}")


def sync_to_py(dry_run=True):
    """%%module 셀을 src/*.py 로 되쓴다. 최종 채택 시에만 사용 (계획 v4 §10)."""
    nb = _json.loads(NB_PATH.read_text(encoding="utf-8"))
    for c in nb["cells"]:
        src = "".join(c["source"])
        if c["cell_type"] != "code" or not src.startswith("%%module "):
            continue
        first, _, body = src.partition("\n")
        _, name, relpath = first.split()
        target = REPO_ROOT / "src" / relpath
        old = target.read_text(encoding="utf-8") if target.exists() else None
        if old == body:
            print(f"  same: {relpath}")
        elif dry_run:
            print(f"CHANGED: {relpath}  (dry_run — 반영하려면 sync_to_py(dry_run=False))")
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_text(body, encoding="utf-8")
            print(f"WROTE: {relpath}")


## 모듈 (graph·vector 노트북과 동일 본문)

In [2]:
%%module config config.py
# -*- coding: utf-8 -*-
"""경로·모델 상수. 채권 MVP 범위."""
from pathlib import Path

# config.py 는 src/ 안에 있다. .parent = src/, 한 번 더 올려야 저장소 루트다.
# 한 번만 올리면 ARTIFACTS 가 src/artifacts 를 새로 만들며 조용히 캐시를 잃는다.
ROOT = Path(__file__).resolve().parent.parent

# 지시서는 ontology/bond.ttl 하나를 가정하지만 이 저장소의 스키마는 common + 도메인 4로 갈려 있다.
# 테스트 질문의 '위험등급'(fp:RiskGrade·fp:riskGradeLevel)은 common.ttl에만 있어서
# bond_kr.ttl만 인덱싱하면 4문항 중 1문항을 못 답한다. 둘 다 넣는다.
BOND_TTL_PATHS = [ROOT / "ontology" / "bond_kr.ttl", ROOT / "ontology" / "common.ttl"]

ARTIFACTS = ROOT / "artifacts"

# 스키마 벡터 인덱스는 PostgreSQL + pgvector 에 둔다(FAISS 에서 이전, 2026-08-22).
# 이전 근거는 vectordb_test/results/1_pgvector_test_report.md — cosine 점수가
# FAISS(정규화 후 IndexFlatIP)와 최대 오차 5.03e-07 로 일치해 임계값을 그대로 쓴다.
# 비밀번호를 포함하므로 DSN 을 로그에 찍지 않는다. 접속 정보는 환경변수로 덮을 수 있다.
import os

# Azure Data API
# 현재 공개 테스트 API는 임시 주소다. 운영에서는 환경변수로 반드시 덮어쓴다.
FINANCIAL_DATA_API_URL = os.environ.get(
    "FINANCIAL_DATA_API_URL",
    "http://40.82.145.44:8000",
)
FINANCIAL_DATA_RELEASE_ID = os.environ.get(
    "FINANCIAL_DATA_RELEASE_ID",
    "financial-products-2026-08-24@"
    "ddb3d994a4a5115a75bed7efa9c4cd0f6655f95b0a49f3b0e3c01b2bf8301a38",
)
DATA_API_TIMEOUT_SECONDS = float(
    os.environ.get("DATA_API_TIMEOUT_SECONDS", "10")
)

# Direct PostgreSQL connection.
# Existing rdb/bond_schema tools still use this configuration.
BOND_DB = {
    "host": os.environ.get("PGHOST", "127.0.0.1"),
    "port": os.environ.get("PGPORT", "5432"),
    "user": os.environ.get("PGUSER", "postgres"),
    "password": os.environ.get("PGPASSWORD", "postgres"),
    "dbname": os.environ.get("PGDATABASE", "mafest"),
}
# 유닉스 소켓은 peer 인증에 걸린다. host 를 명시해 TCP 로 붙는다.
BOND_DSN = " ".join(f"{k}={v}" for k, v in BOND_DB.items())
BOND_TABLE = "bond_schema_terms"
EMBED_DIM = 1024

BOND_TOP_K = 5
# cosine 점수 하한. 실측상 0.36~0.38대는 무관한 용어(자회사 관계 등)가 섞인다.
# 빈약한 근거를 주면 모델이 일반 지식으로 메워 근거 없는 단정이 나온다.
BOND_SCORE_FLOOR = 0.45

EMBEDDING_MODEL = "bge-m3"        # 1024차원, cosine (CLOVA Studio)
# 1단계 Query Frame 추출. HCX-005·HCX-DASH-002 와 대표 4문항으로 비교해 정했다 —
# 스키마 준수 4/4 vs 2/4 vs 1/4. DASH-002 의 속도 이점은 프롬프트가 길어지면서
# 사라졌다(출력 토큰이 지연을 지배한다). 근거: vectordb_test/4_query_frame_v1/4_result_query_frame_v1.md
FRAME_MODEL = "HCX-007"
ANSWER_MODEL = "HCX-005"          # 답변 생성
CHAT_TIMEOUT_SECONDS = 13           # API tail stall은 재시도 없이 ABSTAIN해 15초 E2E를 지킨다

CLOVA_HOST = "https://clovastudio.stream.ntruss.com"


registered: config


In [3]:
%%module clova clova.py
# -*- coding: utf-8 -*-
"""CLOVA Studio 클라이언트 — 채팅과 임베딩.

콘솔 샘플 코드와 다른 점 셋 (실측으로 확인):
  1. Accept를 application/json 으로. text/event-stream 이면 SSE라 반환값으로 못 쓴다.
  2. 추론 모델(HCX-007)은 maxTokens 를 거부한다. maxCompletionTokens 를 쓴다
     — 값 범위 문제가 아니라 파라미터명이 다르다(maxTokens=4096도 40001).
  3. HCX-007에서 structured outputs 를 쓰려면 thinking 을 명시적으로 꺼야 한다.
     기본 ON 이라 responseFormat 과 충돌한다. "off"는 무효값이고 "none"만 받는다.
"""
import json
import sys

import requests

from config import CHAT_TIMEOUT_SECONDS, CLOVA_HOST, EMBEDDING_MODEL, ROOT

REASONING_MODELS = {"HCX-007"}


def load_key() -> str:
    """.env의 clova 키. 값은 절대 로그에 남기지 않는다."""
    env = ROOT / ".env"
    if not env.exists():
        sys.exit(f"FAIL  .env 없음: {env}")
    for line in env.read_text(encoding="utf-8").splitlines():
        k, _, v = line.partition("=")
        if k.strip() == "clova":
            key = v.strip().strip('"').strip("'")
            if not key:
                sys.exit("FAIL  .env의 clova 값이 비어 있음")
            return key if key.startswith("Bearer ") else f"Bearer {key}"
    sys.exit("FAIL  .env에 clova 항목 없음")


_KEY = None


def key() -> str:
    global _KEY
    if _KEY is None:
        _KEY = load_key()
    return _KEY


def chat(model, system, user, max_tokens=1024, response_format=None, temperature=0.1):
    body = {
        "messages": [
            {"role": "system", "content": [{"type": "text", "text": system}]},
            {"role": "user", "content": [{"type": "text", "text": user}]},
        ],
        "topP": 0.8, "temperature": temperature, "repetitionPenalty": 1.1,
        "stop": [], "seed": 0,
    }
    if model in REASONING_MODELS:
        body["maxCompletionTokens"] = max_tokens
        if response_format:
            body["thinking"] = {"effort": "none"}
    else:
        body["maxTokens"] = max_tokens
        body["topK"] = 0
        body["includeAiFilters"] = True
    if response_format:
        body["responseFormat"] = response_format

    r = requests.post(f"{CLOVA_HOST}/v3/chat-completions/{model}",
                      headers={"Authorization": key(),
                               "Content-Type": "application/json; charset=utf-8",
                               "Accept": "application/json"},
                      json=body, timeout=CHAT_TIMEOUT_SECONDS)
    if r.status_code != 200:
        # 본문을 삼키면 원인을 못 찾는다. 40001 메시지에 어느 파라미터인지 들어 있다.
        raise RuntimeError(f"HTTP {r.status_code} — {r.text[:220]}")
    data = r.json()
    code = (data.get("status") or {}).get("code")
    if code not in (None, "20000"):
        raise RuntimeError(f"status {code} — {(data.get('status') or {}).get('message')}")
    content = (data.get("result") or {}).get("message", {}).get("content")
    if isinstance(content, list):   # v3는 입력이 배열이라 출력도 배열로 오는 경우가 있다
        content = "".join(p.get("text", "") for p in content if isinstance(p, dict))
    if not isinstance(content, str):
        raise RuntimeError(f"content 형태 불명: {type(content)}")
    return content


def parse_json_loose(text: str) -> dict:
    """모델이 코드펜스나 설명을 붙여도 JSON 객체만 뽑는다."""
    s = text.strip()
    if s.startswith("```"):
        s = s.split("```")[1] if "```" in s[3:] else s[3:]
        s = s.removeprefix("json").strip()
    a, b = s.find("{"), s.rfind("}")
    if a == -1 or b == -1:
        raise json.JSONDecodeError("객체를 못 찾음", s, 0)
    return json.loads(s[a:b + 1])


_EMB = None


def _embedder():
    global _EMB
    if _EMB is None:
        from langchain_naver import ClovaXEmbeddings
        _EMB = ClovaXEmbeddings(model=EMBEDDING_MODEL,
                                api_key=key().removeprefix("Bearer ").strip())
    return _EMB


def _embed_raw(text: str) -> list[float]:
    """캐시를 거치지 않는 실제 API 호출. 이 함수만 embed_query를 부른다.

    embed_many가 이걸 부르고 embed는 embed_many에 위임한다. 셋 중 하나라도
    서로를 부르면 무한 재귀가 된다 — 실제로 embed_many가 embed를 부르던 시절
    캐시 미스에서 RecursionError가 났다.
    """
    return _embedder().embed_query(text)


def embed(text: str) -> list[float]:
    """단건 임베딩. embed_many에 위임해 디스크 캐시를 공유한다.

    직접 embed_query를 부르면 캐시를 지나치므로, 같은 질문이 반복될 때마다
    API를 다시 때리고 간격도 없어 연속 호출 시 429에 걸린다.
    임베딩은 같은 텍스트·모델이면 결정적이라 캐시해도 값이 달라지지 않는다.
    """
    return embed_many([text], pause=0.0, progress=False)[0]


def embed_many(texts: list[str], pause: float = 1.2, progress: bool = True) -> list[list[float]]:
    """여러 건 임베딩. 분당 쿼터가 있어 간격을 두고, 결과는 디스크에 캐시한다.

    - langchain의 embed_documents는 지연 없이 연속 호출해 429(rate exceeded)를 맞는다.
      실측상 약 60건 연속이면 차단되므로 기본 간격을 1.2s(≈50건/분)로 둔다.
    - 캐시가 없으면 중간에 실패할 때 앞서 성공한 호출이 통째로 버려진다.
      TTL 주석은 앞으로 계속 손볼 예정이라 재빌드가 반복된다.
    """
    import hashlib
    import time as _t

    from config import ARTIFACTS
    ARTIFACTS.mkdir(exist_ok=True)
    cache_path = ARTIFACTS / "embed_cache.json"
    cache = {}
    if cache_path.exists():
        try:
            cache = json.loads(cache_path.read_text(encoding="utf-8"))
        except Exception:
            cache = {}      # 깨진 캐시는 조용히 버린다. 다시 만들면 그만이다.

    def kk(t):
        return hashlib.sha1(f"{EMBEDDING_MODEL}\x00{t}".encode()).hexdigest()

    out, new_hits, api_calls = [], 0, 0
    for i, t in enumerate(texts):
        h = kk(t)
        if h in cache:
            out.append(cache[h])
            continue
        for attempt in range(7):
            try:
                v = _embed_raw(t)      # embed() 를 부르면 여기로 되돌아와 무한 재귀가 된다
                break
            except Exception as e:
                if "429" not in str(e) and "42901" not in str(e):
                    raise
                wait = min(2 ** attempt, 65)      # 분 단위 쿼터라 최대 65s까지 기다린다
                if progress:
                    print(f"  429 — {wait}s 대기 ({i+1}/{len(texts)})", flush=True)
                _t.sleep(wait)
        else:
            cache_path.write_text(json.dumps(cache, ensure_ascii=False), encoding="utf-8")
            raise RuntimeError(f"429 재시도 7회 실패: {i}번째 (여기까지는 캐시에 저장됨)")
        out.append(v)
        cache[h] = v
        new_hits += 1
        api_calls += 1
        if new_hits % 20 == 0:      # 중간 저장 — 크래시해도 진행분이 남는다
            cache_path.write_text(json.dumps(cache, ensure_ascii=False), encoding="utf-8")
        if progress and (i + 1) % 25 == 0:
            print(f"  {i+1}/{len(texts)}  (API 호출 {api_calls})", flush=True)
        _t.sleep(pause)

    cache_path.write_text(json.dumps(cache, ensure_ascii=False), encoding="utf-8")
    if progress:
        print(f"  임베딩 완료 — API 호출 {api_calls}건 / 캐시 재사용 {len(texts)-api_calls}건")
    return out


if __name__ == "__main__":
    # 자기검사: embed / embed_many / _embed_raw 의 호출 고리가 닫히지 않았는지 본다.
    # 캐시 적중 경로에서는 재귀가 드러나지 않으므로 반드시 '캐시에 없는' 문장을 쓴다.
    # (실제로 embed_many 가 embed 를 부르던 시절 RecursionError 가 났고,
    #  회귀 테스트 72건이 전부 캐시 적중이라 그 버그를 못 잡았다.)
    import uuid
    probe = f"clova 자기검사 {uuid.uuid4()}"
    v = embed(probe)
    assert len(v) == 1024, f"차원 이상: {len(v)}"
    assert embed(probe) == v, "같은 문장인데 결과가 다르다 — 캐시가 안 먹는다"
    print(f"clova 자기검사 PASS — 캐시미스 경로 dim={len(v)}, 재호출 일치")


registered: clova


In [4]:
%%module tools.graph tools/graph.py
# -*- coding: utf-8 -*-
"""읽기 전용 pyoxigraph SPARQL과 최초 관계 vertical slice."""
from __future__ import annotations

import re
from functools import lru_cache

from config import ARTIFACTS

try:
    from pyoxigraph import Store
except ImportError as exc:  # pragma: no cover - 설치 안내 경로
    raise SystemExit("pyoxigraph 미설치 — python3 -m pip install -r requirements.txt") from exc

STORE_PATH = ARTIFACTS / "oxigraph"
MAX_ROWS = 10_000
_FORBIDDEN = re.compile(
    r"\b(?:ADD|CLEAR|COPY|CREATE|DELETE|DROP|INSERT|LOAD|MOVE|SERVICE|WITH)\b",
    re.IGNORECASE,
)


@lru_cache(maxsize=1)
def _store() -> Store:
    if not STORE_PATH.is_dir():
        raise RuntimeError("Graph store 미구축 — python3 src/kb/build_graph.py")
    return Store.read_only(str(STORE_PATH))


def _value(term):
    if term is None:
        return None
    return term.value


def sparql(query: str) -> bool | list[dict]:
    """SELECT/ASK만 허용한다. Agent는 아래 고정 template 함수만 호출한다."""
    text = query.lstrip()
    text = re.sub(r"(?is)^(?:PREFIX\s+\w*:\s*<[^>]+>\s*)+", "", text).lstrip()
    kind = text.split(None, 1)[0].upper() if text else ""
    if kind not in {"SELECT", "ASK"} or _FORBIDDEN.search(query):
        raise ValueError("Graph query는 SERVICE 없는 SELECT/ASK만 허용합니다")
    result = _store().query(query)
    if kind == "ASK":
        return bool(result)
    variables = [v.value for v in result.variables]
    rows = []
    for solution in result:
        if len(rows) >= MAX_ROWS:
            raise ValueError(f"Graph 결과가 상한 {MAX_ROWS:,}행을 초과했습니다")
        rows.append({name: _value(solution[name]) for name in variables})
    return rows


ECOPRO_HOLDING_QUERY = """
PREFIX fp: <http://mafest.ai/product#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT DISTINCT ?etf ?etf_name ?child ?child_name ?security ?weight ?holding_as_of
                ?holding_source ?relation_as_of ?relation_source
                ?holding_document_title ?holding_document_publisher
                ?holding_document_date ?holding_document_quote
                ?relation_document_title ?relation_document_publisher
                ?relation_document_date ?relation_document_quote WHERE {
  ?parent a fp:Company ; rdfs:label "에코프로" ; fp:hasSubsidiary ?relation .
  ?relation fp:subsidiaryCompany ?child ; fp:asOf ?relation_as_of ;
            fp:sourceId ?relation_source ; fp:supportedBy ?relation_document .
  ?relation_document a fp:Document ; fp:documentTitle ?relation_document_title ;
            fp:documentPublisher ?relation_document_publisher ;
            fp:documentPublishedDate ?relation_document_date ;
            fp:documentQuote ?relation_document_quote .
  FILTER (?relation_as_of <= "2026-07-11"^^xsd:date)
  ?child rdfs:label ?child_name .
  ?security fp:issuedByCompany ?child .
  ?holding fp:holdingSecurity ?security ; fp:asOf ?holding_as_of ;
           fp:sourceId ?holding_source ; fp:supportedBy ?holding_document .
  ?holding_document a fp:Document ; fp:documentTitle ?holding_document_title ;
           fp:documentPublisher ?holding_document_publisher ;
           fp:documentPublishedDate ?holding_document_date ;
           fp:documentQuote ?holding_document_quote .
  FILTER (?holding_as_of <= "2026-07-11"^^xsd:date)
  OPTIONAL { ?holding fp:weight ?weight }
  ?etf a fp:ETF ; fp:hasHolding ?holding ; rdfs:label ?etf_name .
}
ORDER BY ?etf_name ?child_name
"""


def ecopro_subsidiary_etfs() -> list[dict]:
    return sparql(ECOPRO_HOLDING_QUERY)


def evidence_coverage() -> dict:
    """Store 전체의 운영 대상 Graph 관계 evidence 계약을 집계한다."""
    result = {}
    for label, cls in (("holding", "Holding"), ("subsidiary_relation", "SubsidiaryRelation")):
        total = sparql(f"""
PREFIX fp: <http://mafest.ai/product#>
SELECT (COUNT(DISTINCT ?relation) AS ?count) WHERE {{ ?relation a fp:{cls} . }}
""")[0]["count"]
        supported = sparql(f"""
PREFIX fp: <http://mafest.ai/product#>
SELECT (COUNT(DISTINCT ?relation) AS ?count) WHERE {{
  ?relation a fp:{cls} ; fp:supportedBy ?document .
  ?document a fp:Document .
}}
""")[0]["count"]
        total, supported = int(total), int(supported)
        result[label] = {"total": total, "supported": supported,
                         "coverage": supported / total if total else 1.0}
    return result



# === 실험 확장 (EXP-20260828-graph-01) — 채택 전까지 노트북에만 존재 ===
# 아래 템플릿의 어휘(predicate·URI 패턴)는 로컬 store(1,628,311 triples, cutoff 2026-08-24)
# 실측 프로브로 검증된 것이다:
#   상품: fpi:{etfkr|etfgl|fund|bond}-<코드>, fp:productShortName/productName/productCode
#   기업: fpi:corp-<코드>, fp:organizationName + rdfs:label + skos:altLabel
#   편입: 상품 -fp:hasHolding-> Holding{holdingSecurity, weight, asOf, sourceId, supportedBy}
#   자회사: 기업 -fp:hasSubsidiary-> SubsidiaryRelation{subsidiaryCompany, ownershipPct, asOf, supportedBy}
#   증권↔기업: ?sec fp:issuedByCompany ?corp / 채권 발행: ?bond fp:issuedBy ?issuer
PREFIXES = (
    "PREFIX fp: <http://mafest.ai/product#>\n"
    "PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n"
    "PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\n"
    "PREFIX skos: <http://www.w3.org/2004/02/skos/core#>\n"
    "PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>\n"
)
DATA_CUTOFF = "2026-08-24"


def _lit(text: str) -> str:
    return '"' + str(text).replace("\\", "\\\\").replace('"', '\\"') + '"'


def _q(body: str) -> list:
    return sparql(PREFIXES + body)


def _result(rows, status=None, **extra):
    out = {"status": status or ("ok" if rows else "empty"), "rows": rows}
    out.update(extra)
    return out


def product_info(short_name: str) -> dict:
    """G01/G02: 상품 URI·정식명·코드·투자지역."""
    rows = _q(f"""
SELECT ?product ?name ?code ?region_label WHERE {{
  ?product fp:productShortName {_lit(short_name)} .
  OPTIONAL {{ ?product fp:productName ?name }}
  OPTIONAL {{ ?product fp:productCode ?code }}
  OPTIONAL {{ ?product fp:hasInvestmentRegion ?r . ?r rdfs:label ?region_label .
             FILTER(lang(?region_label) = "ko") }}
}}""")
    return _result(rows)


def product_classifications(short_name: str) -> dict:
    """G08: 상품에 연결된 분류 개체(투자지역·자산유형·테마·위험등급 등)."""
    rows = _q(f"""
SELECT ?pred ?node ?node_label ?node_type WHERE {{
  ?product fp:productShortName {_lit(short_name)} ; ?pred ?node .
  ?node rdf:type ?node_type .
  FILTER(?node_type IN (fp:InvestmentRegion, fp:AssetType, fp:Theme, fp:RiskGrade,
                        fp:FundType, fp:Currency, fp:ManagementStrategy, fp:LeverageType))
  OPTIONAL {{ ?node rdfs:label ?node_label . FILTER(lang(?node_label) = "ko") }}
}}""")
    return _result(rows)


def company_info(name: str) -> dict:
    """G03: 기업 URI + 등록된 다른 이름(rdfs:label, skos:altLabel)."""
    rows = _q(f"""
SELECT ?company ?label ?alt WHERE {{
  ?company rdf:type fp:Company ; fp:organizationName {_lit(name)} .
  OPTIONAL {{ ?company rdfs:label ?label }}
  OPTIONAL {{ ?company skos:altLabel ?alt }}
}}""")
    return _result(rows)


def subsidiaries(company_name: str, limit: int = 30) -> dict:
    """G04: 자회사 관계 + 기준일(asOf)·출처(sourceId, supportedBy 문서 제목)."""
    rows = _q(f"""
SELECT ?relation ?child_name ?ownership_pct ?as_of ?source ?doc_title WHERE {{
  ?parent rdf:type fp:Company ; fp:organizationName {_lit(company_name)} ;
          fp:hasSubsidiary ?relation .
  ?relation fp:subsidiaryCompany ?child .
  ?child fp:organizationName ?child_name .
  OPTIONAL {{ ?relation fp:ownershipPct ?ownership_pct }}
  OPTIONAL {{ ?relation fp:asOf ?as_of }}
  OPTIONAL {{ ?relation fp:sourceId ?source }}
  OPTIONAL {{ ?relation fp:supportedBy ?doc . ?doc fp:documentTitle ?doc_title }}
  FILTER(!BOUND(?as_of) || ?as_of <= "{DATA_CUTOFF}"^^xsd:date)
}} ORDER BY ?child_name LIMIT {int(limit)}""")
    return _result(rows)


def product_holdings(short_name: str, limit: int = 10) -> dict:
    """G05: 상품 → 편입 증권 + 비중 + 기준일."""
    rows = _q(f"""
SELECT ?security_label ?weight ?as_of ?source WHERE {{
  ?product fp:productShortName {_lit(short_name)} ; fp:hasHolding ?h .
  ?h fp:holdingSecurity ?sec .
  OPTIONAL {{ ?sec rdfs:label ?security_label }}
  OPTIONAL {{ ?h fp:weight ?weight }}
  OPTIONAL {{ ?h fp:asOf ?as_of }}
  OPTIONAL {{ ?h fp:sourceId ?source }}
  FILTER(!BOUND(?as_of) || ?as_of <= "{DATA_CUTOFF}"^^xsd:date)
}} ORDER BY DESC(?weight) LIMIT {int(limit)}""")
    return _result(rows)


def etfs_holding_security(security_label: str, limit: int = 20) -> dict:
    """G06/G12: 증권 라벨 → 역방향 → 편입 ETF."""
    rows = _q(f"""
SELECT DISTINCT ?etf ?etf_name ?weight ?as_of WHERE {{
  ?sec rdf:type fp:Security ; rdfs:label {_lit(security_label)} .
  ?h fp:holdingSecurity ?sec .
  ?etf rdf:type fp:ETF ; fp:hasHolding ?h ; fp:productShortName ?etf_name .
  OPTIONAL {{ ?h fp:weight ?weight }}
  OPTIONAL {{ ?h fp:asOf ?as_of }}
}} ORDER BY DESC(?weight) LIMIT {int(limit)}""")
    return _result(rows)


def subsidiary_holding_etfs(company_name: str, limit: int = 50) -> dict:
    """G07: 기업 → 자회사 → (자회사 발행 증권) → 편입 ETF. (ecopro 하드코딩 일반화)"""
    rows = _q(f"""
SELECT DISTINCT ?etf_name ?child_name ?security_label ?weight ?holding_as_of WHERE {{
  ?parent rdf:type fp:Company ; fp:organizationName {_lit(company_name)} ;
          fp:hasSubsidiary ?relation .
  ?relation fp:subsidiaryCompany ?child .
  ?child fp:organizationName ?child_name .
  ?sec fp:issuedByCompany ?child .
  OPTIONAL {{ ?sec rdfs:label ?security_label }}
  ?h fp:holdingSecurity ?sec .
  OPTIONAL {{ ?h fp:weight ?weight }}
  OPTIONAL {{ ?h fp:asOf ?holding_as_of }}
  ?etf rdf:type fp:ETF ; fp:hasHolding ?h ; fp:productShortName ?etf_name .
  FILTER(!BOUND(?holding_as_of) || ?holding_as_of <= "{DATA_CUTOFF}"^^xsd:date)
}} ORDER BY ?etf_name LIMIT {int(limit)}""")
    return _result(rows)


def bond_info(product_name: str) -> dict:
    """G09: 채권 종류(rdf:type)·발행사·신용등급."""
    rows = _q(f"""
SELECT ?bond ?cls ?issuer_name ?rating WHERE {{
  ?bond fp:productName {_lit(product_name)} ; rdf:type ?cls .
  FILTER(?cls IN (fp:CorporateBond, fp:GovernmentBond, fp:SpecialBond, fp:Bond))
  OPTIONAL {{ ?bond fp:issuedBy ?issuer . ?issuer fp:organizationName ?issuer_name }}
  OPTIONAL {{ ?bond fp:hasCreditRating ?r . BIND(REPLACE(STR(?r), ".*#Rating_", "") AS ?rating) }}
}}""")
    return _result(rows)


def fund_holdings_check() -> dict:
    """G10: 펀드 편입 데이터 적재 여부 — empty(관계 없음)와 data_gap(미적재)을 구분."""
    funds = int(_q("SELECT (COUNT(DISTINCT ?f) AS ?n) WHERE { ?f rdf:type fp:PublicFund . ?f fp:hasHolding ?h }")[0]["n"])
    products = int(_q("SELECT (COUNT(DISTINCT ?p) AS ?n) WHERE { ?p fp:hasHolding ?h }")[0]["n"])
    total_funds = int(_q("SELECT (COUNT(DISTINCT ?f) AS ?n) WHERE { ?f rdf:type fp:PublicFund }")[0]["n"])
    status = "ok" if funds else ("data_gap" if products else "empty")
    note = (f"편입 관계 보유 상품 {products}개(전부 ETF), 펀드 {total_funds}개 중 0개 → "
            "펀드 편입 데이터 미적재(data_gap). '펀드가 주식을 편입하지 않는다'가 아니다."
            if status == "data_gap" else "")
    return {"status": status, "rows": [], "funds_with_holdings": funds,
            "products_with_holdings": products, "total_funds": total_funds, "note": note}


def domain_violation(short_name: str, prop: str = "issuedBy") -> dict:
    """G11: TBox rdfs:domain 과 주어 클래스 비교 — 잘못된 온톨로지 관계 거부."""
    subj = _q(f"SELECT DISTINCT ?product ?cls WHERE {{ ?product fp:productShortName {_lit(short_name)} ; rdf:type ?cls }}")
    if not subj:
        return {"status": "empty", "rows": [], "note": "주어 상품 부재"}
    domains = [d["d"] for d in _q(f"SELECT ?d WHERE {{ fp:{prop} rdfs:domain ?d }}")]
    ok_rows = []
    for s in subj:
        for dom in domains:
            if sparql(PREFIXES + f"ASK {{ <{s['cls']}> rdfs:subClassOf* <{dom}> }}"):
                ok_rows.append({"cls": s["cls"], "domain": dom})
    if ok_rows:
        return {"status": "ok", "rows": ok_rows}
    comment = _q(f"SELECT ?c WHERE {{ fp:{prop} rdfs:comment ?c }}")
    return {"status": "abstain_domain_error", "rows": [],
            "subject_classes": sorted({s["cls"] for s in subj}),
            "required_domain": domains,
            "tbox_comment": (comment[0]["c"][:300] if comment else "")}


def product_info_by_code(code: str) -> dict:
    """통합실험 Q02: productCode 로 상품 조회 (shortName 이 모호한 펀드용)."""
    rows = _q(f"""
SELECT ?product ?short_name ?name ?region_label ?risk WHERE {{
  ?product fp:productCode {_lit(code)} .
  OPTIONAL {{ ?product fp:productShortName ?short_name }}
  OPTIONAL {{ ?product fp:productName ?name }}
  OPTIONAL {{ ?product fp:hasInvestmentRegion ?r . ?r rdfs:label ?region_label .
             FILTER(lang(?region_label) = "ko") }}
  OPTIONAL {{ ?product fp:hasRiskGrade ?rg . BIND(REPLACE(STR(?rg), ".*#", "") AS ?risk) }}
}}""")
    return _result(rows)


def etfs_holding_security_like(substr: str, limit: int = 30) -> dict:
    """통합실험 Q03: 증권 라벨 부분일치(영문 별칭 대응) → 편입 ETF + 코드."""
    rows = _q(f"""
SELECT DISTINCT ?etf_name ?etf_code ?security_label ?weight ?as_of WHERE {{
  ?sec rdf:type fp:Security ; rdfs:label ?security_label .
  FILTER(CONTAINS(LCASE(STR(?security_label)), LCASE({_lit(substr)})))
  ?h fp:holdingSecurity ?sec .
  ?etf fp:hasHolding ?h ; fp:productShortName ?etf_name .
  OPTIONAL {{ ?etf fp:productCode ?etf_code }}
  OPTIONAL {{ ?h fp:weight ?weight }}
  OPTIONAL {{ ?h fp:asOf ?as_of }}
}} ORDER BY DESC(?weight) LIMIT {int(limit)}""")
    return _result(rows)


def subsidiary_holding_etf_codes(company_name: str, limit: int = 100) -> dict:
    """통합실험 Q05: 자회사 편입 ETF 의 상품코드 목록 (RDB 랭킹 입력)."""
    rows = _q(f"""
SELECT DISTINCT ?etf_name ?etf_code ?child_name WHERE {{
  ?parent rdf:type fp:Company ; fp:organizationName {_lit(company_name)} ;
          fp:hasSubsidiary ?relation .
  ?relation fp:subsidiaryCompany ?child .
  ?child fp:organizationName ?child_name .
  ?sec fp:issuedByCompany ?child .
  ?h fp:holdingSecurity ?sec .
  ?etf rdf:type fp:ETF ; fp:hasHolding ?h ;
       fp:productShortName ?etf_name ; fp:productCode ?etf_code .
}} ORDER BY ?etf_name LIMIT {int(limit)}""")
    return _result(rows)


def class_counts() -> dict:
    """체크리스트 3번: 주식·ETF·펀드·채권·기업이 모두 조회되는가."""
    out = {}
    for cls in ("Security", "ETF", "PublicFund", "CorporateBond",
                "GovernmentBond", "SpecialBond", "Company"):
        out[cls] = int(_q(f"SELECT (COUNT(?s) AS ?n) WHERE {{ ?s rdf:type fp:{cls} }}")[0]["n"])
    return out


registered: tools.graph


In [5]:
%%module tools.data_api tools/data_api.py
# -*- coding: utf-8 -*-
"""로컬 pgvector 콘텐츠 인덱스(vec.document_chunk) 적재·검색.

계획서의 `tools.data_api.FinancialDataClient` 는 Azure Data API 클라이언트로 문서에만 있고
구현이 없다. Azure `/db` 는 2026-08-29 만료 + 원격 vec 테이블 0행이므로(사용자 결정)
동일한 호출 형태(`FinancialDataClient.from_env().semantic_search(vec, top_k)`)를
로컬 pgvector 로 구현한다. DDL 은 Azure vec.document_chunk 12컬럼을 미러링한다.
"""
import hashlib
from pathlib import Path

import psycopg

import clova
from config import BOND_DSN

EMBED_DIM = 1024
SCORE_FLOOR = 0.45
DATA_CUTOFF = "2026-08-24"

DDL = """
CREATE SCHEMA IF NOT EXISTS vec;
CREATE EXTENSION IF NOT EXISTS vector;
CREATE TABLE IF NOT EXISTS vec.document_chunk (
  chunk_id        text PRIMARY KEY,
  document_id     text NOT NULL,
  product_id      text,
  page_number     integer,
  citation_text   text NOT NULL,
  chunk_text      text NOT NULL,
  published_at    date NOT NULL CHECK (published_at <= DATE '2026-08-24'),
  source_url      text NOT NULL,
  content_hash    text NOT NULL,
  embedding_model text NOT NULL,
  embedding_dim   smallint NOT NULL,
  embedding       vector(1024) NOT NULL
);
CREATE INDEX IF NOT EXISTS document_chunk_embedding_hnsw
  ON vec.document_chunk USING hnsw (embedding vector_cosine_ops);
"""

PRODUCT_TABLES = {"fund_pub": ("raw.fund_pub_master", "itm_no"),
                  "etf_kr": ("raw.etf_kr_master", "pd_itm_no")}


def _conn():
    return psycopg.connect(BOND_DSN, autocommit=True)


def _pages(pdf_path: Path) -> list:
    from pypdf import PdfReader
    reader = PdfReader(str(pdf_path))
    return [(page.extract_text() or "").strip() for page in reader.pages]


def _chunks(text: str, limit: int = 2000, piece: int = 1200) -> list:
    # ponytail: 1페이지=1청크, 2000자 초과만 문단 경계 분할. 검색 품질 부족이 실측되면 슬라이딩 윈도우.
    if len(text) <= limit:
        return [text] if text else []
    out, buf = [], ""
    for para in text.split("\n"):
        if len(buf) + len(para) + 1 > piece and buf:
            out.append(buf.strip())
            buf = ""
        buf += para + "\n"
    if buf.strip():
        out.append(buf.strip())
    return out


def _assert_product(cur, meta):
    """상품코드가 로컬 RDB 에 실재하는지 검증 — 조용한 오매핑 방지 (빌드 게이트)."""
    table, col = PRODUCT_TABLES[meta["domain"]]
    cur.execute(f"SELECT count(*) FROM {table} WHERE {col} = %s", (meta["product_id"],))
    n = cur.fetchone()[0]
    assert n > 0, f"RDB 에 없는 상품코드: {meta['product_id']} ({table}.{col})"


def ingest(pdf_dir, manifest: dict) -> dict:
    """manifest: {파일명: {product_id, domain, published_at}}. TRUNCATE 후 재적재(멱등)."""
    rows = []
    with _conn() as conn, conn.cursor() as cur:
        cur.execute(DDL)
        for fname, meta in manifest.items():
            path = Path(pdf_dir) / fname
            assert path.is_file(), f"PDF 없음: {path}"
            assert str(meta["published_at"]) <= DATA_CUTOFF, f"look-ahead: {fname}"
            _assert_product(cur, meta)
            doc_id = path.stem
            for pno, page_text in enumerate(_pages(path), start=1):
                for i, chunk in enumerate(_chunks(page_text)):
                    rows.append({
                        "chunk_id": f"{doc_id}:p{pno}:{i}",
                        "document_id": doc_id,
                        "product_id": meta["product_id"],
                        "page_number": pno,
                        "citation_text": f"{doc_id} p.{pno}",
                        "chunk_text": chunk,
                        "published_at": meta["published_at"],
                        "source_url": path.resolve().as_uri(),
                        "content_hash": hashlib.sha256(chunk.encode()).hexdigest(),
                    })
        assert rows, "추출된 텍스트 청크가 없다 — PDF 텍스트 추출 실패 여부를 확인할 것"
        vectors = clova.embed_many([r["chunk_text"] for r in rows])
        cur.execute("TRUNCATE vec.document_chunk")
        cur.executemany(
            "INSERT INTO vec.document_chunk (chunk_id, document_id, product_id, page_number,"
            " citation_text, chunk_text, published_at, source_url, content_hash,"
            " embedding_model, embedding_dim, embedding)"
            " VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,'bge-m3',1024,%s::vector)",
            [(r["chunk_id"], r["document_id"], r["product_id"], r["page_number"],
              r["citation_text"], r["chunk_text"], r["published_at"], r["source_url"],
              r["content_hash"], str([float(x) for x in v]))
             for r, v in zip(rows, vectors)])
        cur.execute("SELECT count(*), count(DISTINCT document_id),"
                    " max(vector_dims(embedding)) FROM vec.document_chunk")
        n, docs, dim = cur.fetchone()
    assert n == len(rows) and dim == EMBED_DIM, (n, dim)
    return {"chunks": n, "documents": docs, "dim": dim}


def index_count() -> int:
    try:
        with _conn() as conn, conn.cursor() as cur:
            cur.execute("SELECT count(*) FROM vec.document_chunk")
            return cur.fetchone()[0]
    except psycopg.errors.UndefinedTable:
        return 0


class FinancialDataClient:
    """계획서 호출 형태 유지 어댑터 — 로컬 pgvector 조회."""

    @classmethod
    def from_env(cls):
        return cls()

    def semantic_search(self, query_vector, top_k: int = 3) -> dict:
        vec_literal = str([float(x) for x in query_vector])
        try:
            with _conn() as conn, conn.cursor() as cur:
                cur.execute("SELECT count(*) FROM vec.document_chunk")
                if cur.fetchone()[0] == 0:
                    return {"status": "pending", "results": [], "raw_top": [],
                            "reason": "content index 미구축 — 근거 없음"}
                cur.execute(
                    "SELECT document_id, product_id, page_number, citation_text, chunk_text,"
                    " published_at, 1 - (embedding <=> %(v)s::vector) AS score"
                    " FROM vec.document_chunk"
                    " ORDER BY embedding <=> %(v)s::vector LIMIT %(k)s",
                    {"v": vec_literal, "k": top_k})
                raw = [{"document_id": r[0], "product_id": r[1], "page_number": r[2],
                        "title": r[3], "quote": r[4][:300],
                        "published_at": str(r[5]), "effective_as_of": str(r[5]),
                        "score": round(float(r[6]), 4)}
                       for r in cur.fetchall()]
        except psycopg.errors.UndefinedTable:
            return {"status": "pending", "results": [], "raw_top": [],
                    "reason": "vec.document_chunk 없음"}
        except psycopg.Error as exc:
            return {"status": "error", "results": [], "raw_top": [], "reason": str(exc)}
        hits = [h for h in raw if h["score"] >= SCORE_FLOOR]
        return {"status": "ok" if hits else "empty", "results": hits,
                "raw_top": [{"document_id": h["document_id"], "score": h["score"]} for h in raw]}


registered: tools.data_api


# Q02: 독립 병렬 검색 (Graph ∥ Vector)

In [6]:
# Q02: 독립 병렬 검색 — Graph 상품 속성 ∥ Vector 정책 문서 (계획 v4 §7)
import time
from concurrent.futures import ThreadPoolExecutor

import clova
from tools import graph as G
from tools.data_api import FinancialDataClient

client = FinancialDataClient.from_env()
Q02 = "국민성장펀드의 구조와 투자전략 동향 등 찾아서 알려줘"
Q02_CODE = "KR5153480100"  # 국민참여형 국민성장펀드 대표 클래스(종류C) — RDB·Graph 실재 확인됨


def q02_graph():
    info = G.product_info_by_code(Q02_CODE)
    short = (info["rows"][0].get("short_name") if info["rows"] else None)
    cls = G.product_classifications(short) if short else {"status": "empty", "rows": []}
    return {"info": info, "cls": cls}


def q02_vector():
    return client.semantic_search(clova.embed(Q02), top_k=3)


t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=2) as ex:
    f_g, f_v = ex.submit(q02_graph), ex.submit(q02_vector)
    g_res, v_res = f_g.result(timeout=30), f_v.result(timeout=60)
q02_elapsed = round((time.perf_counter() - t0) * 1000, 1)

# 결과 병합 — 근거가 있는 내용만 (규칙 4)
q02_evidence = []
for r in g_res["info"]["rows"]:
    q02_evidence.append({"source": "graph", "subject": r["product"],
                         "predicate": "productName/region/riskGrade",
                         "object": f"{r.get('name')} | {r.get('region_label')} | {r.get('risk')}",
                         "as_of": "2026-08-21 (fund_pub 실질 기준일)"})
for r in g_res["cls"]["rows"]:
    q02_evidence.append({"source": "graph", "subject": Q02_CODE, "predicate": r["pred"],
                         "object": r.get("node_label") or r["node"], "as_of": None})
for h in v_res.get("results", []):
    q02_evidence.append({"source": "vector", "text": h["quote"], "score": h["score"],
                         "document_id": h["document_id"], "as_of": h["published_at"]})

q02_record = {
    "question_id": "Q02", "question": Q02,
    "mode": "parallel(graph, vector)",
    "graph_status": g_res["info"]["status"], "vector_status": v_res["status"],
    "status": "ok" if q02_evidence else "empty",
    "evidence_n": len(q02_evidence), "elapsed_ms": q02_elapsed,
}
print(q02_record)
for e in q02_evidence[:6]:
    print(" ", {k: (str(v)[:80] if v else v) for k, v in e.items()})

{'question_id': 'Q02', 'question': '국민성장펀드의 구조와 투자전략 동향 등 찾아서 알려줘', 'mode': 'parallel(graph, vector)', 'graph_status': 'ok', 'vector_status': 'ok', 'status': 'ok', 'evidence_n': 9, 'elapsed_ms': 2741.1}
  {'source': 'graph', 'subject': 'http://mafest.ai/instance/fund-KR5153480100', 'predicate': 'productName/region/riskGrade', 'object': '미래에셋국민참여형국민성장혼합자산투자신탁(사모투자재간접형) 종류C (일반형) | 국내 | RiskGrade_1', 'as_of': '2026-08-21 (fund_pub 실질 기준일)'}
  {'source': 'graph', 'subject': 'KR5153480100', 'predicate': 'http://mafest.ai/product#hasFundType', 'object': '혼합자산형', 'as_of': None}
  {'source': 'graph', 'subject': 'KR5153480100', 'predicate': 'http://mafest.ai/product#hasRiskGrade', 'object': '매우 높은 위험(1등급)', 'as_of': None}
  {'source': 'graph', 'subject': 'KR5153480100', 'predicate': 'http://mafest.ai/product#hasInvestmentRegion', 'object': '국내', 'as_of': None}
  {'source': 'graph', 'subject': 'KR5153480100', 'predicate': 'http://mafest.ai/product#hasCurrency', 'object': '한국 원', 'as_of': None}


# Q03: Graph 또는 Graph → RDB

In [7]:
# Q03: Graph 또는 Graph → RDB — 실측 전에 경로를 확정하지 않는다 (계획 v4 §7)
import psycopg

from config import BOND_DSN

Q03 = "캠브리콘이 편입된 중국 반도체 ETF를 알려줘"
# 그래프에 '캠브리콘' 한글 라벨은 없다(실측) — 증권 라벨은 영문뿐이므로 한→영 별칭이 필요하다.
ALIAS = {"캠브리콘": "Cambricon"}  # ponytail: 별칭 2~3개 dict. 늘어나면 RDB 영문명 매핑으로 교체

t0 = time.perf_counter()
q03_hits = G.etfs_holding_security_like(ALIAS["캠브리콘"])
etf_names = sorted({r["etf_name"] for r in q03_hits["rows"]})

# 중국·반도체 조건 확인 1차: Graph 분류(Theme·InvestmentRegion)
cond_in_graph = {}
for name in etf_names:
    cls = G.product_classifications(name)
    labels = " ".join(str(r.get("node_label") or r.get("node")) for r in cls["rows"])
    cond_in_graph[name] = {"china": ("중국" in labels or "차이나" in labels or "China" in labels),
                           "semi": ("반도체" in labels)}

# Graph 에 조건이 없는 ETF는 RDB(relations.etf_theme)로 2차 확인 → graph_then_rdb
need_rdb = [n for n, c in cond_in_graph.items() if not (c["china"] and c["semi"])]
rdb_cond = {}
if need_rdb:
    codes = sorted({r["etf_code"] for r in q03_hits["rows"]
                    if r.get("etf_code") and r["etf_name"] in need_rdb})
    with psycopg.connect(BOND_DSN) as conn, conn.cursor() as cur:
        cur.execute("SET TRANSACTION READ ONLY")
        cur.execute("SELECT pd_itm_no, string_agg(theme, ',') FROM relations.etf_theme"
                    " WHERE pd_itm_no = ANY(%s) GROUP BY pd_itm_no", (codes,))
        themes = dict(cur.fetchall())
    for r in q03_hits["rows"]:
        if r["etf_name"] in need_rdb and r.get("etf_code") in themes:
            t = themes[r["etf_code"]]
            rdb_cond[r["etf_name"]] = {"china": ("중국" in t or "차이나" in t), "semi": "반도체" in t}
q03_elapsed = round((time.perf_counter() - t0) * 1000, 1)

matched = [n for n in etf_names
           if (cond_in_graph[n]["china"] and cond_in_graph[n]["semi"])
           or (rdb_cond.get(n, {}).get("china") and rdb_cond.get(n, {}).get("semi"))]
route = "graph_only" if matched and not rdb_cond else ("graph_then_rdb" if matched else "graph_then_rdb(조건 미충족)")

q03_record = {
    "question_id": "Q03", "question": Q03, "route_observed": route,
    "holding_etfs": etf_names, "matched_china_semi": matched,
    "cond_in_graph": cond_in_graph, "cond_in_rdb": rdb_cond,
    "status": "ok" if matched else "empty", "elapsed_ms": q03_elapsed,
}
print("route_observed:", route)
print("편입 ETF:", etf_names)
print("중국+반도체 충족:", matched)

route_observed: graph_then_rdb
편입 ETF: ['ACE 글로벌자율주행액티브', 'ACE 중국과창판STAR50', 'ACE 중국본토CSI300', 'ACE 차이나AI빅테크TOP2+액티브', 'KODEX 차이나A50', 'KODEX 차이나AI반도체TOP10', 'KODEX 차이나AI테크액티브', 'KODEX 차이나CSI300', 'RISE 중국본토CSI300', 'RISE 중국본토대형주CSI100', 'RISE 차이나AI반도체TOP4Plus', 'TIGER 차이나CSI300', 'TIGER 차이나반도체FACTSET', 'TIGER 차이나테크TOP10']
중국+반도체 충족: ['KODEX 차이나AI반도체TOP10', 'RISE 차이나AI반도체TOP4Plus', 'TIGER 차이나반도체FACTSET']


# Q05: 순차 검색 (Graph → RDB → Vector)

In [8]:
# Q05: 순차 검색 — 에코프로 →(Graph) 자회사 →(Graph) 편입 ETF →(RDB) 순자산 최대 →(Vector) 위험요인
# 앞 단계 결과가 다음 단계 조건이므로 병렬로 실행하지 않는다 (계획 v4 §7).
Q05 = "에코프로의 자회사를 편입한 ETF 중 순자산이 큰 상품의 위험요인 알려줘"

t0 = time.perf_counter()
step1 = G.subsidiaries("에코프로")                       # Graph: 자회사
step2 = G.subsidiary_holding_etf_codes("에코프로")        # Graph: 자회사 편입 ETF + 코드
codes = sorted({r["etf_code"] for r in step2["rows"] if r.get("etf_code")})

# RDB: 순자산(pd_net_tamt, 실질 기준일 2026-08-21) 최대 — 후보 IN-list 랭킹은
# LogicalPlan compiler 미지원이라 실험에서는 직접 read-only SQL 로 확인한다.
top_etf = None
if codes:
    with psycopg.connect(BOND_DSN) as conn, conn.cursor() as cur:
        cur.execute("SET TRANSACTION READ ONLY")
        cur.execute("SELECT pd_itm_no, pd_abrv_nm, pd_net_tamt, du_nav_base_dt"
                    " FROM raw.etf_kr_master WHERE pd_itm_no = ANY(%s)"
                    " ORDER BY pd_net_tamt DESC NULLS LAST LIMIT 1", (codes,))
        row = cur.fetchone()
        if row:
            top_etf = {"code": row[0], "name": row[1], "net_assets": str(row[2]),
                       "as_of": row[3], "source": "raw.etf_kr_master.pd_net_tamt"}

# Vector: 위험요인 문서 — 검색 결과의 product_id 가 해당 ETF 와 다르면 근거로 쓰지 않는다 (규칙 4)
risk = {"status": "skipped", "results": []}
risk_evidence = []
if top_etf:
    risk = client.semantic_search(clova.embed(f"{top_etf['name']} ETF 투자 위험요인"), top_k=3)
    risk_evidence = [h for h in risk.get("results", []) if h.get("product_id") == top_etf["code"]]
q05_elapsed = round((time.perf_counter() - t0) * 1000, 1)

q05_record = {
    "question_id": "Q05", "question": Q05, "mode": "sequential(graph→graph→rdb→vector)",
    "subsidiaries_n": len(step1["rows"]), "candidate_etfs_n": len(codes),
    "candidate_codes": codes[:20], "top_etf": top_etf,
    "vector_status": risk["status"], "risk_evidence_n": len(risk_evidence),
    "risk_note": ("" if risk_evidence else
                  "해당 ETF 의 위험요인 문서가 콘텐츠 인덱스에 미적재 → 이 부분은 '확인할 수 없음' (data_gap)"),
    "status": "partial" if (top_etf and not risk_evidence) else ("ok" if risk_evidence else "empty"),
    "elapsed_ms": q05_elapsed,
}
for k, v in q05_record.items():
    print(f"{k}: {v}")

question_id: Q05
question: 에코프로의 자회사를 편입한 ETF 중 순자산이 큰 상품의 위험요인 알려줘
mode: sequential(graph→graph→rdb→vector)
subsidiaries_n: 22
candidate_etfs_n: 45
candidate_codes: ['KR70094M0005', 'KR7102960002', 'KR7156080004', 'KR7229200001', 'KR7232080002', 'KR7233160001', 'KR7233740000', 'KR7270810005', 'KR7275280006', 'KR7278540000', 'KR7279540009', 'KR7289040008', 'KR7289250003', 'KR7292050002', 'KR7292160009', 'KR7292190006', 'KR7305540007', 'KR7305720005', 'KR7306950007', 'KR7310970009']
top_etf: {'code': 'KR7229200001', 'name': 'KODEX 코스닥150', 'net_assets': '5180073904940.00', 'as_of': '20260821', 'source': 'raw.etf_kr_master.pd_net_tamt'}
vector_status: empty
risk_evidence_n: 0
risk_note: 해당 ETF 의 위험요인 문서가 콘텐츠 인덱스에 미적재 → 이 부분은 '확인할 수 없음' (data_gap)
status: partial
elapsed_ms: 9934.6


# 기록 저장 + 실험 종료 후 비교 (계획 v4 §8·§9)

In [9]:
import json

records = [q02_record, q03_record, q05_record]
out_path = RESULTS_DIR / "integrated_0828.json"
out_path.write_text(json.dumps({"experiment_id": "EXP-20260828-integrated-01",
                                "records": records},
                               ensure_ascii=False, indent=2, default=str),
                    encoding="utf-8")
print("saved:", out_path)

# 실험 종료 후 비교 (계획 v4 §9)
def load(name):
    p = RESULTS_DIR / name
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else None

graph_res = load("graph_0828.json")
vector_res = load("vector_0828.json")
main_res = load("main_baseline_0828.json")

print()
print("=== 계획 v4 §9 비교 ===")
if graph_res:
    ok = [r["question_id"] for r in graph_res["records"] if r["verdict"] == "PASS" and r["status"] == "ok"]
    print("1. 답변 가능한 Graph 질문:", ok)
if vector_res:
    print("2. 문서를 검색한 Vector 질문:",
          [r["question_id"] for r in vector_res if r["status"] == "ok"])
if graph_res:
    g04 = next(r for r in graph_res["records"] if r["question_id"] == "G04")
    print("3. Graph 결과 문서 근거(supportedBy):",
          any(row.get("doc_title") for row in g04.get("rows", [])))
if vector_res:
    v_ok = [r for r in vector_res if r["status"] == "ok"]
    print("4. Vector 결과 기준일·출처:",
          all(h.get("published_at") and h.get("title") for r in v_ok
              for h in r["retrieved_documents"]))
print("5. Q03 조건 확인 저장소:", q03_record["route_observed"])
print("6. Q05 순차 실행 가능:", bool(q05_record["top_etf"]))
if main_res:
    print("7. 기존 RDB 결과 유지:",
          [ (r["question_id"], (r["validation"] or {}).get("code", "OK")) for r in main_res["records"] ])

saved: /mnt/c/Users/rladl/Desktop/2026_MIRAE_ASSET_AI-Festival/2026_10th_MIRAE-ASSET_AI-Festival/test/notebook/experiments/results/integrated_0828.json

=== 계획 v4 §9 비교 ===
1. 답변 가능한 Graph 질문: ['G01', 'G02', 'G03', 'G04', 'G05', 'G06', 'G07', 'G08', 'G09']
2. 문서를 검색한 Vector 질문: ['V01', 'V02', 'V03', 'V04', 'V05', 'V06', 'V07']
3. Graph 결과 문서 근거(supportedBy): True
4. Vector 결과 기준일·출처: True
5. Q03 조건 확인 저장소: graph_then_rdb
6. Q05 순차 실행 가능: True
7. 기존 RDB 결과 유지: [('Q01', 'ABSTAIN_UNRESOLVED_QUERY'), ('Q06', 'ABSTAIN_INVALID_TAXONOMY'), ('Q08', 'ABSTAIN_ENTITY_NOT_FOUND')]
